In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os
import json
import gc
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
IMAGE_DIR    = "./images"          # folder containing raw images
OUTPUT_DIR   = "./sam_output"      # where per-image JSON will be saved
SAM_WEIGHTS  = "./sam_weights/sam_vit_h_4b8939.pth"
MODEL_TYPE   = "vit_h"             # must match the weight file

# ─── MASK GENERATOR PARAMETERS ────────────────────────────────────────────────
SAM_PARAMS = dict(
    points_per_side          = 32,
    points_per_batch         = 16,
    pred_iou_thresh          = 0.90,   # raised: filter out lower-quality masks
    stability_score_thresh   = 0.96,   # raised: require more stable boundaries
    stability_score_offset   = 1.0,
    box_nms_thresh           = 0.50,   # more aggressive NMS → fewer overlapping masks
    crop_n_layers            = 0,
    min_mask_region_area     = 3000,   # min object size in pixels²
)

# ─── POST-FILTER ──────────────────────────────────────────────────────────────
MIN_AREA_PX        = 3000   # mirrors min_mask_region_area
MAX_AREA_PX        = None   # set e.g. 0.5 * W * H to remove background masks
MAX_MASKS_PER_IMG  = 150    # hard cap: keep top-N masks ranked by predicted_iou

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Images             : {IMAGE_DIR}")
print(f"Output             : {OUTPUT_DIR}")
print(f"Weights            : {SAM_WEIGHTS}")
print(f"CUDA               : {torch.cuda.is_available()}")
print(f"Max masks/image    : {MAX_MASKS_PER_IMG}")


In [ ]:
def load_sam_model(weights_path: str, model_type: str, device: str = "cuda"):
    """Load SAM model onto the specified device."""
    print(f"Loading SAM ({model_type}) from {weights_path} ...")
    sam = sam_model_registry[model_type](checkpoint=weights_path)
    sam.to(device=device)
    sam.eval()
    print("SAM loaded successfully.")
    return sam

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sam_model = load_sam_model(SAM_WEIGHTS, MODEL_TYPE, DEVICE)

In [ ]:
def mask_to_polygons(binary_mask: np.ndarray, simplify_tolerance: float = 1.5):
    """
    Convert a binary mask (H×W uint8) to a list of polygons.
    Each polygon is a flat list [x1,y1, x2,y2, ...] for CVAT compatibility.

    Returns a list of polygons (one mask can have multiple contours
    if it contains holes or disconnected regions — we keep only the
    largest contour per mask to stay clean).
    """
    # Ensure uint8
    mask_uint8 = (binary_mask > 0).astype(np.uint8) * 255

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,      # only outer contours — ignores holes
        cv2.CHAIN_APPROX_SIMPLE # compress straight edges
    )

    if not contours:
        return []

    # Keep only the largest contour (avoids tiny noise fragments)
    largest = max(contours, key=cv2.contourArea)

    # Douglas-Peucker simplification to reduce vertex count
    epsilon = simplify_tolerance
    approx = cv2.approxPolyDP(largest, epsilon, closed=True)

    # Need at least 3 points for a valid polygon
    if len(approx) < 3:
        return []

    # Flatten to [x1, y1, x2, y2, ...]
    points = approx.reshape(-1, 2).tolist()
    flat   = [coord for point in points for coord in point]
    return [flat]  # list of polygons (one per contour kept)


def get_image_files(image_dir: str):
    """Return sorted list of image paths from a directory."""
    supported = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
    paths = sorted([
        p for p in Path(image_dir).iterdir()
        if p.suffix.lower() in supported
    ])
    return paths

> ⚠️ **WAJIB sebelum menjalankan cell ini:** Jika kamu baru saja mengedit parameter di cell sebelumnya (SAM_PARAMS, MAX_MASKS_PER_IMG, dll), pastikan kamu **Restart Kernel dan Run All Cells** (`Kernel → Restart & Run All`).
> Mengedit file `.ipynb` **tidak** memperbarui fungsi yang sudah berjalan di kernel — sehingga perubahan parameter tidak akan aktif sampai cell di-run ulang dari awal.

In [ ]:
def process_single_image(
    image_path: Path,
    mask_generator: SamAutomaticMaskGenerator,
    min_area: int,
    max_area,
    max_masks: int,
    simplify_tolerance: float = 4.0
):
    """
    Run SAM AutomaticMaskGenerator on a single image.

    Filtering order:
      1. Remove masks below min_area / above max_area
      2. Sort remaining masks by predicted_iou (descending)
      3. Keep only top `max_masks` before polygon conversion (performance)
      4. Final hard cap after polygon conversion (safety guarantee)
    """
    # ─ Read image ────────────────────────────────────────────────────────────
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        print(f"  [WARN] Could not read {image_path.name}, skipping.")
        return None

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]

    # ─ Generate masks ─────────────────────────────────────────────────────────
    masks = mask_generator.generate(rgb)

    # ─ Step 1: Area filter ────────────────────────────────────────────────────
    filtered = []
    for m in masks:
        area = int(m["area"])
        if area < min_area:
            continue
        if max_area is not None and area > max_area:
            continue
        filtered.append(m)

    # ─ Step 2: Sort by predicted_iou descending, keep top max_masks ──────────
    # This runs BEFORE polygon conversion to avoid wasting time on discarded masks.
    filtered.sort(key=lambda m: m["predicted_iou"], reverse=True)
    filtered = filtered[:max_masks]

    # ─ Step 3: Convert to polygons ───────────────────────────────────────────
    records = []
    for m in filtered:
        polys = mask_to_polygons(
            m["segmentation"].astype(np.uint8),
            simplify_tolerance=simplify_tolerance
        )
        if not polys:
            continue

        records.append({
            "area"            : int(m["area"]),
            "bbox"            : [int(v) for v in m["bbox"]],
            "predicted_iou"   : float(m["predicted_iou"]),
            "stability_score" : float(m["stability_score"]),
            "polygons"        : polys
        })

    # ─ Step 4: Final hard cap — guarantees output never exceeds max_masks ────
    # Defensive layer: ensures the limit holds even if kernel has stale
    # function definitions from a previous run without the top-K logic.
    if len(records) > max_masks:
        records.sort(key=lambda r: r["predicted_iou"], reverse=True)
        records = records[:max_masks]

    return {
        "filename" : image_path.name,
        "width"    : w,
        "height"   : h,
        "n_masks"  : len(records),
        "masks"    : records
    }


def run_pipeline(
    image_dir    : str,
    output_dir   : str,
    sam_model,
    sam_params   : dict,
    min_area     : int,
    max_area,
    max_masks    : int   = 150,
    skip_existing: bool  = True,
    simplify_tol : float = 4.0
):
    """
    Process all images in image_dir sequentially.

    Memory strategy:
      - MaskGenerator is constructed once and reused.
      - After each image: torch.cuda.empty_cache() + gc.collect()
    """
    image_paths = get_image_files(image_dir)
    print(f"Found {len(image_paths)} images in '{image_dir}'")
    print(f"Hard cap per image : {max_masks} masks (ranked by predicted_iou)")

    mask_generator = SamAutomaticMaskGenerator(sam_model, **sam_params)

    results = {"processed": 0, "skipped": 0, "failed": 0}

    for img_path in tqdm(image_paths, desc="Processing images"):
        out_path = Path(output_dir) / (img_path.stem + ".json")

        if skip_existing and out_path.exists():
            results["skipped"] += 1
            continue

        try:
            record = process_single_image(
                img_path, mask_generator,
                min_area, max_area, max_masks, simplify_tol
            )

            if record is None:
                results["failed"] += 1
                continue

            with open(out_path, "w") as f:
                json.dump(record, f)

            results["processed"] += 1
            tqdm.write(
                f"  {img_path.name}: {record['n_masks']} masks "
                f"({record['width']}×{record['height']})"
            )

        except Exception as e:
            tqdm.write(f"  [ERROR] {img_path.name}: {e}")
            results["failed"] += 1

        finally:
            torch.cuda.empty_cache()
            gc.collect()

    print("\n──── Run Summary ────")
    print(f"  Processed : {results['processed']}")
    print(f"  Skipped   : {results['skipped']}  (already done)")
    print(f"  Failed    : {results['failed']}")
    return results


# ─── RUN ──────────────────────────────────────────────────────────────────────
run_pipeline(
    image_dir     = IMAGE_DIR,
    output_dir    = OUTPUT_DIR,
    sam_model     = sam_model,
    sam_params    = SAM_PARAMS,
    min_area      = MIN_AREA_PX,
    max_area      = MAX_AREA_PX,
    max_masks     = MAX_MASKS_PER_IMG,
    skip_existing = True,
    simplify_tol  = 4.0
)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

def visualize_result(image_dir, output_dir, filename, max_display=150):
    """Overlay SAM polygons on the original image for verification."""
    img_path = Path(image_dir) / filename
    json_path = Path(output_dir) / (Path(filename).stem + ".json")

    bgr = cv2.imread(str(img_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    with open(json_path) as f:
        record = json.load(f)

    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    ax.imshow(rgb)

    patches = []
    for i, mask in enumerate(record["masks"][:max_display]):
        for poly_pts in mask["polygons"]:
            pts = np.array(poly_pts).reshape(-1, 2)
            patch = MplPolygon(pts, closed=True)
            patches.append(patch)

    collection = PatchCollection(
        patches,
        alpha=0.35,
        facecolor=np.random.rand(len(patches), 3),
        edgecolor="white",
        linewidth=0.5
    )
    ax.add_collection(collection)
    ax.set_title(f"{filename}  —  {record['n_masks']} masks detected", fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# Replace with one of your actual image filenames
visualize_result(IMAGE_DIR, OUTPUT_DIR, "img_001.jpg")